# Evaluate Synthetic Title Baselines

Use this notebook to evaluate OCR and LayoutLMv3 baselines on a generated dataset split.

It reuses the repo's shared experiment utilities, so the notebook and CLI scripts compute the same metrics.

In [ ]:
from pathlib import Path
import json

from experiment_utils import compute_ocr_metrics, evaluate_model_on_examples, load_examples

## Parameters

Set the dataset path, split, and OCR source here.

In [ ]:
DATASET_DIR = Path("../../dataset")
MODEL_DIR = Path("models/layoutlmv3-funsd")
SPLIT = "test"              # train | val | test
SEED = 42
MAX_SAMPLES = None          # set to an int for quick checks
OCR_SOURCE = "tesseract"    # tesseract | paddleocr | ground_truth
DEVICE = None               # None -> auto-select cpu/cuda
OUTPUT_PATH = None          # e.g. Path("../../runs/baselines/notebook_eval.json")

In [ ]:
examples = load_examples(
    dataset_dir=DATASET_DIR,
    split=SPLIT,
    seed=SEED,
    max_samples=MAX_SAMPLES,
)

print(f"Loaded {len(examples)} examples from {DATASET_DIR} split={SPLIT}")
examples[:2]

In [ ]:
ocr_backend = "tesseract" if OCR_SOURCE == "ground_truth" else OCR_SOURCE
ocr_metrics = compute_ocr_metrics(examples, ocr_source=ocr_backend)
layoutlm_metrics = evaluate_model_on_examples(
    examples=examples,
    model_dir=MODEL_DIR,
    ocr_source=OCR_SOURCE,
    device=DEVICE,
)

result = {
    "dataset_dir": str(DATASET_DIR),
    "model_dir": str(MODEL_DIR),
    "split": SPLIT,
    "num_examples": len(examples),
    "ocr_metrics": ocr_metrics,
    "layoutlm_metrics": layoutlm_metrics,
}

print(json.dumps(result, indent=2))

In [ ]:
if OUTPUT_PATH is not None:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(OUTPUT_PATH, "w") as fh:
        json.dump(result, fh, indent=2)
    print(f"Saved metrics to {OUTPUT_PATH}")